# Riverside: describe → validate → render → revise

A fictional request-review service. A coordinator registers and checks a request.
Incomplete requests return to the requester for clarification and then repeat review.
Complete requests need **both** technical and finance checks in parallel before the
coordinator issues a response. Explicit handoffs transfer responsibility.

Run with the repository's Python environment (Python 3.14+, FLO and Node.js).
For rich notebook display, install Jupyter/IPython separately; the build script
also runs without them. See `README.md` beside this notebook.


In [ ]:
from pathlib import Path
import importlib.util
import json
import flo
import yaml

# Locate this example from either the repository root or notebook directory.
ROOT = Path.cwd().resolve()
while not (ROOT / "examples/reference/riverside/current.flo").exists():
    if ROOT == ROOT.parent:
        raise RuntimeError("Start this notebook inside the FLO checkout")
    ROOT = ROOT.parent
EXAMPLE = ROOT / "examples/reference/riverside"
spec = importlib.util.spec_from_file_location("riverside", EXAMPLE / "run.py")
riverside = importlib.util.module_from_spec(spec)
spec.loader.exec_module(riverside)
source_path = EXAMPLE / "current.flo"
source = source_path.read_text(encoding="utf-8")
print(source)


## Validate and read the findings

Errors stop publication. Warnings explain the limits of a valid model.
`timing-incomplete` means evidence is missing: collect the listed fields or keep
them unknown. `timing-parallel-flow-unsupported` means FLO cannot calculate a
path lead time for this split/join. Never replace missing values with zero to
make a warning disappear. Source references are opaque; FLO does not fetch them.


In [ ]:
validated = flo.validate(source, source_path=str(source_path))
assert validated.ok, validated.report.as_dict()
for finding in validated.diagnostics:
    print(f"{finding.code}: {finding.message}")
    if finding.details:
        print(json.dumps(finding.details, indent=2))


## Practice a repair

This exercise changes only an in-memory copy. The primary selector must name a
result in the profile. Read the error's field path, line and excerpt; repair
`missing` to `mean` and validate again. Other useful exercises: remove a unit,
misspell a transition target, or remove a required parallel join.


In [ ]:
model = yaml.safe_load(source)
model["steps"][1]["cycle_time"]["primary"] = "missing"
broken = flo.validate(yaml.safe_dump(model, sort_keys=False), source_path="exercise.flo")
assert not broken.ok
for finding in broken.report.as_dict()["diagnostics"]:
    print(json.dumps(finding, indent=2))
print("Action: set steps[1].cycle_time.primary to mean, an existing result name.")
model["steps"][1]["cycle_time"]["primary"] = "mean"
assert flo.validate(yaml.safe_dump(model, sort_keys=False)).ok


## Render and compare

Current uses synthetic summaries (`basis: simulated`), not field observations.
Proposed halves known waits using a shared checklist and review alerts; these
are workshop estimates with unknown observation counts. The flow, processing
assumptions, step IDs, mean statistic and minute units stay consistent.
Counts refer to activity executions, including repeated visits, not unique
requests. The three clarification executions have a different cohort from the
12 mainline executions. Their means cannot be treated as a process mean.

P/T = processing, W/T = waiting, S/T = setup. The detail tables preserve basis,
counts and source references in notebook and print; diagram labels are compact.


In [ ]:
report = riverside.build(ROOT / "renders/riverside")
try:
    from IPython.display import HTML, display
except ImportError:
    print(report)
else:
    display(HTML(report.read_text(encoding="utf-8")))


## Revise with evidence

Edit `proposed.flo`, rerun validation and rebuild the comparison. Review roles,
both parallel checks, the clarification return, and the detail table together.
The complete report is `renders/riverside/comparison.html`; open it in a browser
and print at A3 portrait, 100% scale. SVG and full JSON remain beside it.

**End-to-end time is unavailable.** Do not sum parallel paths, add all branches,
or count a repeated activity only once and call it lead time. FLO's declared
graph totals are inventories, not elapsed process statistics. A future measured
comparison needs request-level start/end observations, compatible cohorts,
branch frequencies and repeat counts. No savings percentage is justified here.

For real measured data use `basis: observed`, truthful observation counts and a
real evidence reference. Keep synthetic examples labeled; use `estimated` for
assumptions. Missing timing stays omitted; a known zero stays numeric zero.
